# Visualization

Genera todos los gráficos estáticos e interactivos del proyecto.

**Inputs** (de `data/processed/`):
- `distrito_scores.csv` — índice de acceso por distrito (baseline y alternativa)
- `distritos_base.gpkg` — polígonos distritales con datos agregados
- `ipress_clean.csv` — IPRESS con coordenadas para mapas de puntos

**Outputs** (en `output/figures/`):
- `dist_scores.png` — distribución de scores baseline vs alternativa
- `top_bottom.png` — top/bottom 10 distritos por acceso
- `mapa_acceso.png` — mapa coroplético estático de scores
- `mapa_componentes.png` — mapa de los 3 componentes del índice
- `mapa_interactivo.html` — mapa Folium con score + clasificación
- `mapa_ipress.html` — mapa Folium con puntos de IPRESS

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import folium
from pathlib import Path

ROOT          = Path(r"C:\Users\esarmiento\Documents\GitHub\emergency_access_peru")
PROCESSED_DIR = ROOT / "data" / "processed"
FIGURES_DIR   = ROOT / "output" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Scores tabulares
df_scores = pd.read_csv(PROCESSED_DIR / "distrito_scores.csv", dtype={"ubigeo": str})

# Polígonos distritales (geometría necesaria para mapas)
gdf_base = gpd.read_file(PROCESSED_DIR / "distritos_base.gpkg")

# IPRESS con coordenadas (solo columnas necesarias)
df_ipress = pd.read_csv(
    PROCESSED_DIR / "ipress_clean.csv",
    dtype={"ubigeo": str},
    usecols=["codigo_unico", "nombre", "ubigeo", "norte", "este", "nivel"]
).dropna(subset=["norte", "este"])

# Unimos scores con geometría
gdf = gdf_base.merge(df_scores, on="ubigeo", how="left")

print("GDF final:", gdf.shape)
print("Score nulos:", gdf["score_baseline"].isna().sum())
print("IPRESS con coords:", df_ipress.shape)

## Gráfico 1: Distribución del índice de acceso

Histograma superpuesto de scores baseline (5 km) y alternativa (15 km).
Permite ver cómo cambia la distribución al ampliar el umbral de distancia.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

# Histograma baseline
ax.hist(
    df_scores["score_baseline"].dropna(),
    bins=50, alpha=0.6, color="steelblue",
    label="Baseline (5 km)"
)
# Histograma alternativa
ax.hist(
    df_scores["score_alternativa"].dropna(),
    bins=50, alpha=0.6, color="coral",
    label="Alternativa (15 km)"
)

# Líneas de mediana
ax.axvline(df_scores["score_baseline"].median(), color="steelblue",
           linestyle="--", linewidth=1.5,
           label=f"Mediana baseline = {df_scores['score_baseline'].median():.3f}")
ax.axvline(df_scores["score_alternativa"].median(), color="coral",
           linestyle="--", linewidth=1.5,
           label=f"Mediana alternativa = {df_scores['score_alternativa'].median():.3f}")

ax.set_xlabel("Índice de acceso a emergencias (0–1)", fontsize=11)
ax.set_ylabel("Número de distritos", fontsize=11)
ax.set_title("Distribución del índice de acceso a emergencias por distrito", fontsize=13)
ax.legend(fontsize=9)
sns.despine(ax=ax)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "dist_scores.png", dpi=150)
plt.show()
print("Guardado: dist_scores.png")

## Gráfico 2: Top / Bottom 10 distritos

Barras horizontales con los 10 distritos de mayor y menor acceso según el score baseline.
Permite identificar los extremos de la desigualdad.

In [ ]:
df_valid = df_scores.dropna(subset=["score_baseline"]).copy()

top10 = df_valid.nlargest(10, "score_baseline")[["distrito", "departamen", "score_baseline"]]
bot10 = df_valid.nsmallest(10, "score_baseline")[["distrito", "departamen", "score_baseline"]]

# Etiqueta: Distrito (Dpto)
top10["label"] = top10["distrito"] + "\n(" + top10["departamen"] + ")"
bot10["label"] = bot10["distrito"] + "\n(" + bot10["departamen"] + ")"

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Top 10
axes[0].barh(top10["label"], top10["score_baseline"], color="seagreen", alpha=0.85)
axes[0].set_xlabel("Score baseline", fontsize=10)
axes[0].set_title("Top 10 — Mayor acceso", fontsize=12)
axes[0].invert_yaxis()
sns.despine(ax=axes[0])

# Bottom 10
axes[1].barh(bot10["label"], bot10["score_baseline"], color="tomato", alpha=0.85)
axes[1].set_xlabel("Score baseline", fontsize=10)
axes[1].set_title("Bottom 10 — Menor acceso", fontsize=12)
axes[1].invert_yaxis()
sns.despine(ax=axes[1])

fig.suptitle("Distritos con mayor y menor acceso a emergencias (baseline 5 km)", fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "top_bottom.png", dpi=150)
plt.show()
print("Guardado: top_bottom.png")

## Mapa 1: Coroplético estático — Score baseline

Mapa de Perú coloreado por el índice de acceso a emergencias (baseline 5 km).
Generado con GeoPandas + matplotlib.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 12))

# Distritos sin score en gris
gdf[gdf["score_baseline"].isna()].plot(
    ax=ax, color="#d0d0d0", linewidth=0.1
)

# Distritos con score coloreados
gdf[gdf["score_baseline"].notna()].plot(
    ax=ax,
    column="score_baseline",
    cmap="YlOrRd",
    linewidth=0.1,
    edgecolor="white",
    legend=True,
    legend_kwds={
        "label": "Índice de acceso (0–1)",
        "orientation": "horizontal",
        "shrink": 0.5,
        "pad": 0.01
    }
)

ax.set_title("Índice de acceso a emergencias por distrito\n(Baseline: umbral 5 km)", fontsize=13)
ax.axis("off")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "mapa_acceso.png", dpi=150, bbox_inches="tight")
plt.show()
print("Guardado: mapa_acceso.png")

## Mapa 2: Los 3 componentes del índice

Tres mapas coropléticos lado a lado mostrando cada componente normalizado:
- **Comp 1**: Disponibilidad (IPRESS / centros poblados)
- **Comp 2**: Actividad (atenciones / IPRESS reportante)
- **Comp 3**: Acceso espacial (% CP dentro de 5 km de un IPRESS)

In [ ]:
componentes = [
    ("comp1_norm", "Disponibilidad\n(IPRESS / CP)", "Blues"),
    ("comp2_norm", "Actividad\n(Atenciones / IPRESS)", "Greens"),
    ("comp3_norm", "Acceso espacial\n(% CP a ≤5 km)", "Oranges"),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 10))

for ax, (col, titulo, cmap) in zip(axes, componentes):
    if col not in gdf.columns:
        ax.set_title(f"{titulo}\n(sin datos)")
        ax.axis("off")
        continue

    gdf[gdf[col].isna()].plot(ax=ax, color="#d0d0d0", linewidth=0.1)
    gdf[gdf[col].notna()].plot(
        ax=ax, column=col, cmap=cmap,
        linewidth=0.1, edgecolor="white",
        legend=True,
        legend_kwds={"orientation": "horizontal", "shrink": 0.5, "pad": 0.01}
    )
    ax.set_title(titulo, fontsize=12)
    ax.axis("off")

fig.suptitle("Componentes del índice de acceso a emergencias", fontsize=14)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "mapa_componentes.png", dpi=150, bbox_inches="tight")
plt.show()
print("Guardado: mapa_componentes.png")

## Mapa 3: Interactivo Folium — Clasificación por distrito

Mapa web con polígonos distritales coloreados por clasificación (Subatendido / Acceso medio / Mejor atendido).
Al hacer click en un distrito se muestra el score, clasificación y departamento.

In [ ]:
# Convertimos GDF a WGS84 si es necesario y simplificamos geometría para el HTML
gdf_folium = gdf[["ubigeo", "distrito", "departamen", "score_baseline", "clasificacion", "geometry"]].copy()
gdf_folium = gdf_folium.to_crs(epsg=4326)

# Simplificamos geometría para aligerar el HTML (tolerancia en grados)
gdf_folium["geometry"] = gdf_folium["geometry"].simplify(tolerance=0.01, preserve_topology=True)

# Paleta de colores por clasificación
color_map = {
    "Subatendido":     "#d73027",
    "Acceso medio":    "#fee090",
    "Mejor atendido":  "#4575b4",
}

def estilo(feature):
    clasif = feature["properties"].get("clasificacion", None)
    color = color_map.get(clasif, "#cccccc")
    return {"fillColor": color, "color": "white", "weight": 0.5, "fillOpacity": 0.7}

# Mapa centrado en Perú
m = folium.Map(location=[-9.2, -75.0], zoom_start=5, tiles="CartoDB positron")

folium.GeoJson(
    gdf_folium.__geo_interface__,
    style_function=estilo,
    tooltip=folium.GeoJsonTooltip(
        fields=["distrito", "departamen", "score_baseline", "clasificacion"],
        aliases=["Distrito:", "Departamento:", "Score baseline:", "Clasificación:"],
        localize=True
    )
).add_to(m)

# Leyenda manual
leyenda_html = """
<div style="position:fixed;bottom:30px;left:30px;z-index:1000;
            background:white;padding:10px;border-radius:6px;
            border:1px solid #ccc;font-size:13px;">
  <b>Clasificación de acceso</b><br>
  <span style="background:#d73027;width:12px;height:12px;display:inline-block;"></span> Subatendido<br>
  <span style="background:#fee090;width:12px;height:12px;display:inline-block;"></span> Acceso medio<br>
  <span style="background:#4575b4;width:12px;height:12px;display:inline-block;"></span> Mejor atendido<br>
  <span style="background:#cccccc;width:12px;height:12px;display:inline-block;"></span> Sin datos
</div>
"""
m.get_root().html.add_child(folium.Element(leyenda_html))

output_path = FIGURES_DIR / "mapa_interactivo.html"
m.save(str(output_path))
print("Guardado:", output_path)
m

## Mapa 4: Interactivo Folium — Puntos de IPRESS

Mapa de puntos con cada IPRESS que tiene coordenadas válidas.
Coloreado por nivel de atención (I, II, III, Sin Categoría).

In [ ]:
nivel_colors = {
    "I":              "green",
    "II":             "orange",
    "III":            "red",
    "Sin Categoría":  "gray",
}

m2 = folium.Map(location=[-9.2, -75.0], zoom_start=5, tiles="CartoDB positron")

for _, row in df_ipress.iterrows():
    nivel = str(row.get("nivel", "Sin Categoría"))
    color = nivel_colors.get(nivel, "gray")
    folium.CircleMarker(
        location=[row["este"], row["norte"]],   # este=lat, norte=lon
        radius=3,
        color=color,
        fill=True,
        fill_opacity=0.7,
        popup=folium.Popup(
            f"<b>{row['nombre']}</b><br>Nivel: {nivel}<br>UBIGEO: {row['ubigeo']}",
            max_width=200
        )
    ).add_to(m2)

leyenda2_html = """
<div style="position:fixed;bottom:30px;left:30px;z-index:1000;
            background:white;padding:10px;border-radius:6px;
            border:1px solid #ccc;font-size:13px;">
  <b>Nivel de atención IPRESS</b><br>
  <span style="color:green;">●</span> Nivel I<br>
  <span style="color:orange;">●</span> Nivel II<br>
  <span style="color:red;">●</span> Nivel III<br>
  <span style="color:gray;">●</span> Sin Categoría
</div>
"""
m2.get_root().html.add_child(folium.Element(leyenda2_html))

output_path2 = FIGURES_DIR / "mapa_ipress.html"
m2.save(str(output_path2))
print("Guardado:", output_path2)
m2